# {{PROJECT_TITLE}}

## [Project subtitle]

**Notebook:** {{NOTEBOOK_NAME}}  
**Phase:** {{PHASE}}  
**Source:** [Dataset name, file, row count and original publisher]  
**Author:** {{AUTHOR}}  
**Date:** {{DATE}}  

## {{NOTEBOOK_PURPOSE}}

[One paragraph: what this notebook does, what it reads and what it produces.]

In [ ]:
# Comment each import with what it is used for (Playbook Part V, Code Comment Standard).
import os                            # reads settings from environment variables
from pathlib import Path             # builds file paths from the project root

from dotenv import load_dotenv       # loads the .env file into environment variables

## Configuration

Settings come from `.env`, so no credentials or project-specific IDs are written into the notebook.

In [ ]:
# The notebook runs from notebooks/, so the project root is one folder up.
PROJECT_ROOT = Path.cwd().resolve().parent
load_dotenv(PROJECT_ROOT / ".env")

RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Fixed seed so random steps (sampling, bootstrap, train/test splits) give the same result on every run
RANDOM_SEED = 42

## Checksum Check

Stops the notebook if any raw file differs from the one documented in `data/raw/SHA256SUMS.txt` (Playbook Phase 1 → Data Provenance). Keep this cell in the loading notebook only.

In [ ]:
import hashlib                       # computes SHA-256 checksums

def sha256(path):
    """Checksum read in 1 MB blocks, so large files do not fill memory."""
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

# Each line is "<checksum>  <file path relative to data/raw/>", as written by `shasum -a 256`.
for line in (RAW_DIR / "SHA256SUMS.txt").read_text().splitlines():
    if not line.strip():
        continue
    expected, name = line.split(maxsplit=1)
    actual = sha256(RAW_DIR / name.lstrip("*"))
    assert actual == expected, f"{name}: checksum differs from SHA256SUMS.txt; do not continue"
    print(f"checksum OK  {name}")

## Warehouse Snapshot Check

The data stays in BigQuery, so there is no file to checksum. The first run saves each source table's row count and last-modified time to `data/raw/WAREHOUSE_SNAPSHOT.csv` (commit it). Every later run compares the tables with that snapshot and stops if a table was added, removed or changed size (Playbook Phase 1 → Data Provenance). Keep this cell in the loading notebook only.

In [ ]:
from google.cloud import bigquery    # Google's Python client for BigQuery
import pandas as pd                  # compares the saved snapshot with the current tables

# Project, source dataset and cost cap come from .env, so no IDs are written into the notebook.
client = bigquery.Client(project=os.environ["GCP_PROJECT_ID"])
SOURCE_DATASET = os.environ["BQ_SOURCE_DATASET"]                       # e.g. bigquery-public-data.<dataset>
MAX_BYTES_BILLED = int(os.environ.get("BQ_MAX_BYTES_BILLED", "1000000000"))

def run_query(sql):
    """Run a query with a byte cap: a query that would scan more fails instead of using quota."""
    job_config = bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES_BILLED)
    return client.query(sql, job_config=job_config).to_dataframe()

# __TABLES__ holds table metadata (row counts, last-modified times); reading it does not scan the data.
current = run_query(f"""
    SELECT table_id, row_count, TIMESTAMP_MILLIS(last_modified_time) AS last_modified
    FROM `{SOURCE_DATASET}.__TABLES__`
    ORDER BY table_id
""")
current["last_modified"] = current["last_modified"].astype(str)

SNAPSHOT = RAW_DIR / "WAREHOUSE_SNAPSHOT.csv"
if not SNAPSHOT.exists():
    current.to_csv(SNAPSHOT, index=False)
    print(f"snapshot written  {len(current)} tables, {current['row_count'].sum():,} rows; commit data/raw/WAREHOUSE_SNAPSHOT.csv")
else:
    saved = pd.read_csv(SNAPSHOT, dtype={"table_id": str, "last_modified": str})
    both = saved.merge(current, on="table_id", how="outer", suffixes=("_saved", "_now"), indicator=True)
    # A table added or removed, or a different row count, means the data is not what was documented.
    differs = both[(both["_merge"] != "both") | (both["row_count_saved"] != both["row_count_now"])]
    assert differs.empty, f"{len(differs)} table(s) differ from WAREHOUSE_SNAPSHOT.csv; do not continue:\n{differs}"
    # Same row count but a newer last-modified time: the source was rewritten. Reported, not stopped.
    touched = both[both["last_modified_saved"] != both["last_modified_now"]]
    if not touched.empty:
        print(f"note: {len(touched)} table(s) have a new last-modified time but the same row count")
    print(f"snapshot OK  {len(current)} tables, {current['row_count'].sum():,} rows")

## Figures Quoted

Every figure the README, project page or dashboard text quotes, printed from the results above. Copy figures into the write-up from this output only (Playbook Part V → Figures in Text). Run it last, after a kernel restart.

In [ ]:
# One entry per figure quoted in the write-up. Each value reads from a computed result,
# never a typed number, so a re-run updates the figure. Example:
#     "Churn rate": f"{churn_rate:.2%}",
figures_quoted = {
}

for name, value in figures_quoted.items():
    print(f"{name}: {value}")